# Simplified pie chart — merged genome annotation

Five-category pie of the **newly merged genome annotation** (Liftoff + Braker3/AUGUSTUS merge).

This notebook produces only the simplified pie. The detailed 14-slice pie and all of the
sunbursts (OctDeg1 / LiftOff / Braker / final) live in `annotation_pieChart_visualization.ipynb`.


In [ ]:
# Project root — edit for your environment.
PROJ_ROOT = "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj"


In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio


In [ ]:
gff_cols = ["seqid", "source", "type", "start", "end", "score", "strand", "phase", "attributes"]

# Newly merged genome annotation (Liftoff + Braker3/AUGUSTUS merge).
gff_final = pd.read_csv(
    f"{PROJ_ROOT}/code/command-line-script/annotation-merging/output/hifiasm-041425-denovoEnhanced_peaks2utr_sorted.agat.gff3",
    sep="\t", comment="#", header=None, names=gff_cols, usecols=["type", "attributes"],
)

# Reference (Liftoff-transferred) annotation — used only to decide "novel" vs "characterized".
gff_lo = pd.read_csv(
    f"{PROJ_ROOT}/output/outputs-from-liftoff/hifiasm-041425-scaffolded-chrAssigned-mito/hifiasm-041425-scaffolded-chrAssigned-mito.gff",
    sep="\t", comment="#", header=None, names=gff_cols, usecols=["type", "attributes"],
)


In [ ]:
# Reference gene-name set = protein-coding genes transferred by Liftoff.
# A final gene is "novel" when its (suffix-stripped) parent name is absent from this set.
gff_lo["biotype"] = gff_lo["attributes"].str.extract(r"gene_biotype=([^;]+)", expand=False)
gff_lo["gene_name"] = gff_lo["attributes"].str.extract(r"ID=gene-([^;]+)", expand=False)

reference_gene_names = (
    gff_lo.loc[gff_lo["biotype"] == "protein_coding", "gene_name"]
    .dropna()
    .drop_duplicates()
    .str.upper()
)
len(reference_gene_names)


In [ ]:
# Annotate the final (merged) gene set.
test = gff_final[gff_final["type"] == "gene"].copy()

test["biotype"] = (
    test["attributes"]
    .str.extract(r"gene_biotype=([^;]+)", expand=False)
    .fillna("protein_coding")  # de-novo / replaced genes lack a biotype; treat as protein-coding
)
test["gene_name"] = test["attributes"].str.extract(r"ID=gene-([^;]+)", expand=False).str.upper()

# Parent name = gene name with the paralog-copy suffix (-L1 / -DL1 / -RL1) removed.
test["gene_name_undup"] = test["gene_name"].str.replace(r"-(D?R?L\d+)$", "", regex=True)

test["is_LOC_gene"] = np.where(
    test["gene_name"].str.match(r"^LOC\d{9}$", na=False),
    "LOC genes",
    "Characterized genes",
)

# "novel" = parent gene has no reference (Liftoff) ortholog.
test["new_gene"] = ~test["gene_name_undup"].str.upper().isin(reference_gene_names)


In [ ]:
# One row per protein-coding gene in the final annotation.
plot_final_clean = (
    test[["biotype", "gene_name", "is_LOC_gene", "new_gene"]]
    .dropna()
    .drop_duplicates()
)
plot_final_clean = plot_final_clean[plot_final_clean["biotype"] == "protein_coding"]
len(plot_final_clean)


In [ ]:
# Simplified pie categories, built directly from the paralog-copy (suffix) definition.
# paralog copies = gene name ends in -(D?R?L\d+)$  (e.g. -L1, -DL1, -RL1).
feature_totals = plot_final_clean.copy()
feature_totals["paralog_copy"] = feature_totals["gene_name"].str.contains(r"-(D?R?L\d+)$", na=False, regex=True)

def _ft_label(r):
    if r["is_LOC_gene"] == "LOC genes":
        return "LOC genes"
    if r["paralog_copy"] and r["new_gene"]:
        return "Novel-gene paralog copies"
    if r["paralog_copy"]:
        return "Characterized-gene paralog copies"
    if r["new_gene"]:
        return "Novel genes"
    return "Characterized genes"

feature_totals["label"] = feature_totals.apply(_ft_label, axis=1)
feature_totals = feature_totals.groupby("label").size().reset_index(name="count")

custom_order = [
    "Characterized genes", "Characterized-gene paralog copies","Novel genes",
    "Novel-gene paralog copies",  "LOC genes",
]
feature_totals["label"] = pd.Categorical(feature_totals["label"], categories=custom_order, ordered=True)
feature_totals = feature_totals.sort_values("label")
feature_totals


In [ ]:
labels = feature_totals["label"].astype(str).tolist()
counts = feature_totals["count"].tolist()
colors = ["#98df8a" if l == "LOC genes" else ("#ff9896" if "Novel" in l else "#aec7e8") for l in labels]
pull = [0.12 if "Novel" in l else 0 for l in labels]
pattern_map = {
    "Characterized genes": "",
    "Characterized-gene paralog copies": ".",
    "Novel-gene paralog copies": "x",
    "Novel genes": "/",
    "LOC genes": "",
}
pattern_shape = [pattern_map.get(l, "") for l in labels]

fig = go.Figure(
    data=[go.Pie(labels=labels, values=counts, pull=pull, rotation=90, sort=False,
                 marker=dict(colors=colors, line=dict(color="#000000", width=2),
                             pattern_shape=pattern_shape))],
    layout=go.Layout(width=1200, height=1200, legend=dict(orientation="h")),
)
fig.update_traces(hoverinfo="label+percent", textinfo="value", textfont_size=30)
pio.write_image(fig, "pie_chart_final_simplified_bottemLegend.svg", format="svg", engine="kaleido")
pio.write_image(fig, "pie_chart_final_simplified_bottemLegend.png", scale=3, engine="kaleido")
fig.show()


In [ ]:
labels = feature_totals["label"].astype(str).tolist()
counts = feature_totals["count"].tolist()
colors = ["#98df8a" if l == "LOC genes" else ("#ff9896" if "Novel" in l else "#aec7e8") for l in labels]
pull = [0.12 if "Novel" in l else 0 for l in labels]
pattern_map = {
    "Characterized genes": "",
    "Characterized-gene paralog copies": ".",
    "Novel genes": "",
    "Novel-gene paralog copies": ".",
    "LOC genes": "",
}
pattern_shape = [pattern_map.get(l, "") for l in labels]

fig = go.Figure(
    data=[go.Pie(labels=labels, values=counts, pull=pull, rotation=120, sort=False,
                 marker=dict(colors=colors, line=dict(color="#000000", width=2),
                             pattern_shape=pattern_shape))],
    layout=go.Layout(width=1200, height=1200, legend=dict(orientation="h")),
)
fig.update_traces(hoverinfo="label+percent", textinfo="label+value", textfont_size=25)
pio.write_image(fig, "pie_chart_final_simplified_labeled.svg", format="svg", engine="kaleido")
pio.write_image(fig, "pie_chart_final_simplified_labeled.png", scale=3, engine="kaleido")
fig.show()
